In [ ]:
import pandas as pd
import numpy as np
from tabpfn import TabPFNClassifier, TabPFNRegressor

/opt/anaconda3/envs/general_12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
gold_path = '/Users/ormenessevinicius/Library/CloudStorage/OneDrive-TheBostonConsultingGroup,Inc/Documents/GitHub/analise_times/football_analysis/data/gold_partidas_features/part-0.parquet'
df = pd.read_parquet(gold_path)

In [ ]:
df

,match_id,match_date,year_month,month_index,competition_name,season,source,home_team,away_team,home_score,...,away_GK_onoff_gols_sofridos_mean_18m_q,away_DEF_onoff_gols_sofridos_mean_18m_q,away_MID_onoff_gols_sofridos_mean_18m_q,away_FWD_onoff_gols_sofridos_mean_18m_q,away_onoff_xg_sofrido_mean_18m_q,away_GK_onoff_xg_sofrido_mean_18m_q,away_DEF_onoff_xg_sofrido_mean_18m_q,away_MID_onoff_xg_sofrido_mean_18m_q,away_FWD_onoff_xg_sofrido_mean_18m_q,target
0,statsbomb:69229,2009-09-12,200909,24116,La Liga,2009/2010,statsbomb,Getafe,Barcelona,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L
1,statsbomb:69292,2009-09-19,200909,24116,La Liga,2009/2010,statsbomb,Barcelona,Atlético Madrid,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,W
2,statsbomb:69254,2009-09-22,200909,24116,La Liga,2009/2010,statsbomb,Racing Santander,Barcelona,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L
3,statsbomb:69255,2009-09-26,200909,24116,La Liga,2009/2010,statsbomb,Málaga,Barcelona,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L
4,statsbomb:69256,2009-10-03,200910,24117,La Liga,2009/2010,statsbomb,Barcelona,Almería,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,W
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3771,statsbomb:4018356,2025-07-18,202507,24306,UEFA Women's Euro,2025,statsbomb,Spain Women's,Switzerland Women's,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,W
3772,statsbomb:4018357,2025-07-19,202507,24306,UEFA Women's Euro,2025,statsbomb,France Women's,Germany Women's,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D
3773,statsbomb:4020005,2025-07-22,202507,24306,UEFA Women's Euro,2025,statsbomb,England Women's,Italy Women's,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,W
3774,statsbomb:4020077,2025-07-23,202507,24306,UEFA Women's Euro,2025,statsbomb,Germany Women's,Spain Women's,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L


In [ ]:
do_not_use = [
    'year_month',
    'month_index',
    'home_score',
    'away_score'
]
thresh = len(df) * 0.20
numerical_cols = [i for i in df.dropna(axis=1, thresh=thresh).select_dtypes(include="number").columns.tolist() if i not in do_not_use and 'mean' in i]
len(numerical_cols)

592

In [ ]:
df["target"] = np.where(
    df["home_score"] > df["away_score"],
    "W",
    np.where(
        df["home_score"] < df["away_score"],
        "L",
        "D"
    )
)

In [ ]:
import optuna
import lightgbm as lgb
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

X = df[numerical_cols]
y = df['target']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

def objective(trial):
    params = {
        "objective": "multiclass",
        "num_class": len(le.classes_),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "random_state": 42,
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "n_jobs": 1,  # important on macOS
    }

    model = lgb.LGBMClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(
        model,
        X,
        y_encoded,
        cv=cv,
        scoring="accuracy",
        n_jobs=1  # important on macOS
    )
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, n_jobs=1)  # keep sequential

print("Best score:", study.best_value)
print("Best params:", study.best_params)

best_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(le.classes_),
    metric="multi_logloss",
    boosting_type="gbdt",
    verbosity=-1,
    random_state=42,
    n_jobs=1,
    **study.best_params
)

best_model.fit(X, y_encoded)

[I 2026-06-11 19:01:48,658] A new study created in memory with name: no-name-d9860856-bd97-472d-8d79-8783f33a6d4e


[I 2026-06-11 19:03:45,137] Trial 0 finished with value: 0.5259525561512317 and parameters: {'n_estimators': 371, 'learning_rate': 0.04196455024759541, 'num_leaves': 125, 'max_depth': 9, 'min_child_samples': 67, 'subsample': 0.8538762608184681, 'colsample_bytree': 0.7162171937507604, 'reg_alpha': 0.01877503921084803, 'reg_lambda': 0.24992372239155652}. Best is trial 0 with value: 0.5259525561512317.
[I 2026-06-11 19:04:50,334] Trial 1 finished with value: 0.5455492483969305 and parameters: {'n_estimators': 269, 'learning_rate': 0.022822799341070173, 'num_leaves': 58, 'max_depth': 5, 'min_child_samples': 73, 'subsample': 0.7728563331374049, 'colsample_bytree': 0.9379802988265744, 'reg_alpha': 0.03549516622685306, 'reg_lambda': 0.002278111741792217}. Best is trial 1 with value: 0.5455492483969305.
[I 2026-06-11 19:06:38,066] Trial 2 finished with value: 0.5500525596552086 and parameters: {'n_estimators': 205, 'learning_rate': 0.011489585956173175, 'num_leaves': 123, 'max_depth': 9, 'min_

Best score: 0.5582606258102947
Best params: {'n_estimators': 132, 'learning_rate': 0.014880302863724808, 'num_leaves': 97, 'max_depth': 10, 'min_child_samples': 35, 'subsample': 0.9179193710198861, 'colsample_bytree': 0.7266559979015762, 'reg_alpha': 9.999405398514103, 'reg_lambda': 0.013339623449058813}


,boosting_type,'gbdt'
,num_leaves,97
,max_depth,10
,learning_rate,0.014880302863724808
,n_estimators,132
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,35


In [ ]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df.head(20))

                                      feature  importance
24           away_team_form_xg_total_mean_18m         126
194        away_DEF_setor_key_passes_mean_18m         125
45              home_lineup_xg_total_mean_18m         112
208         away_DEF_setor_def_acoes_mean_18m         110
108          home_DEF_setor_xg_total_mean_18m         101
5          home_team_form_key_passes_mean_18m         100
289     away_MID_onoff_gols_sofridos_mean_18m          99
21               away_team_form_gols_mean_18m          99
268  away_GK_setor_def_gols_sofridos_mean_18m          94
226      away_MID_setor_ev_clearance_mean_18m          93
284        home_MID_onoff_xg_sofrido_mean_18m          93
119       home_DEF_setor_ev_pressure_mean_18m          89
69               away_lineup_ev_shot_mean_18m          87
3            home_team_form_xg_total_mean_18m          87
281            home_onoff_xg_sofrido_mean_18m          87
47            home_lineup_key_passes_mean_18m          86
193  away_DEF_

In [ ]:
print(1)

In [ ]:
import pandas as pd

from world_cup_features import (
    get_training_feature_columns,
    build_team_feature_store,
    prepare_match_row,
    prepare_world_cup_matches,
)

gold = pd.read_parquet("football_analysis/data/gold_partidas_features/part-0.parquet")
fixtures = pd.read_csv("football_analysis/model_notebooks/world_cup_2026_games.csv")

feature_columns = get_training_feature_columns(gold)
store = build_team_feature_store(gold, feature_columns)

# One game
x_mex_sa = prepare_match_row("Mexico", "South Africa", store)
pred_class = best_model.predict(x_mex_sa)
pred_proba = best_model.predict_proba(x_mex_sa)

print(pred_class, pred_proba)

In [ ]:
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np

import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0"

# X = your feature dataframe
# y = target column with values like "W", "D", "L"

X = df[importance_df['feature'].values[:250]]
y = df['target']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Local model: first fit downloads weights once, then runs locally
    clf = TabPFNClassifier.create_default_for_version(
        ModelVersion.V2,
        device="mps"
    )
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )

    rows.append({
        "fold": fold,
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "precision_weighted": p_weighted,
        "recall_weighted": r_weighted,
        "f1_weighted": f1_weighted,
    })

results = pd.DataFrame(rows)
summary = pd.DataFrame({
    "metric": results.drop(columns="fold").columns,
    "mean": results.drop(columns="fold").mean().values,
    "std": results.drop(columns="fold").std().values,
})

print(results)
print("\nAverage CV stats:")
print(summary)

TypeError: str expected, not float